# Native AHR v5 — live progress

Purpose: inspect the 18-hour experiment without starting another writer. Inputs: current curriculum, controller and training reports. Outputs: current stage, skill gate, development/final outcomes and remaining budget in Dublin time. Expected runtime: under ten seconds. No checkpoint is written.

Run All to refresh. Session: `tmux attach -t ahr-go-native-v5`; detach **Ctrl-b, then d**. Main checkpoints: `checkpoints/ahr-native-v5-main/`. Exact resume: notebook 05 with `AHR_NATIVE_V5_RUN=1`, unchanged sources/configuration. The original budget remains in effect. Final results are never used for training.

In [ ]:
from pathlib import Path
import datetime as dt, html, json, subprocess
from zoneinfo import ZoneInfo
from IPython.display import display, HTML
project=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebooks/00_instance_setup_and_recovery.ipynb').is_file())
root=project/'reports/ahr_native_v5'
def read(name):
    p=root/name
    return json.loads(p.read_text()) if p.is_file() else {}
controller=read('pipeline_status.json');training=read('ahr-native-v5-main_training.json');curriculum=read('curriculum_progress.json')
session_exists=subprocess.run(['tmux','has-session','-t','=ahr-go-native-v5'],capture_output=True).returncode==0
panes=subprocess.run(['tmux','list-panes','-s','-t','=ahr-go-native-v5','-F','#{pane_dead}'],capture_output=True,text=True) if session_exists else None
session_live=bool(panes and panes.returncode==0 and '0' in panes.stdout.splitlines())
recovery=read('runtime_gate_recovery.json')
controller_status=controller.get('status','not launched')
if controller_status=='INTERRUPTED_OR_FAILED' and session_live and training.get('status')=='RUNNING' and recovery.get('journal_state')=='COMMITTED' and training.get('optimizer_steps',0)>242:
    controller_status='RUNNING after verified recovery'
now=dt.datetime.now(ZoneInfo('Europe/Dublin'))
def local(unix):return dt.datetime.fromtimestamp(unix,ZoneInfo('Europe/Dublin')).strftime('%a %d %b %H:%M %Z') if unix else 'pending'
gate=training.get('learning_gate',{})
rows=[('Checked',now.strftime('%a %d %b %H:%M:%S %Z')),('Detached session','running' if session_live else 'not running'),
      ('Controller',controller_status),('Current phase',training.get('phase',controller.get('phase','curriculum / validation'))),
      ('Verified curriculum',str(curriculum.get('examples',0))+' tasks; '+curriculum.get('status','pending')),
      ('Optimizer steps',training.get('optimizer_steps',0)),('Training games',training.get('games_finished',0)),('Learning gate',gate.get('status','not measured')),
      ('Training UNKNOWN',training.get('unknown_games',0)),('Learning window ends',local(controller.get('learning_deadline_unix'))),('Hard budget ends',local(controller.get('hard_deadline_unix')))]
def table(headers,values):
    return '<table style="border-collapse:collapse;font-size:15px"><thead><tr>'+''.join('<th style="padding:8px;border-bottom:2px solid #bbb;text-align:left">'+html.escape(str(x))+'</th>' for x in headers)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td style="padding:7px;border-bottom:1px solid #ddd">'+html.escape(str(x))+'</td>' for x in row)+'</tr>' for row in values)+'</tbody></table>'
display(HTML(table(['Item','Current value'],rows)))


In [ ]:
reviews=controller.get('development',[])
if reviews:
    display(HTML(table(['Evaluation','Wins / attempts','Losses','UNKNOWN','Mean score margin'],[(r['eval_run'],str(r['wins'])+' / '+str(r['games_planned']),r['losses'],r['unknown'],round(r['mean_score_margin'],1) if r.get('mean_score_margin') is not None else 'unscored') for r in reviews])))
by_goal=gate.get('by_goal',gate.get('skill',gate.get('tactical',{})).get('by_goal',{}))
if by_goal:
    display(HTML(table(['Bounded task','Held-out success'],[(g,format(v.get('top1_success',0) if isinstance(v,dict) else v,'.1%')) for g,v in sorted(by_goal.items())])))
final=controller.get('final')
if final:display(HTML('<p>Final evaluation: <b>'+html.escape(str(final['wins']))+' wins / '+str(final['games_planned'])+' planned attempts</b>; UNKNOWN '+str(final['unknown'])+'; '+html.escape(final['status'])+'.</p>'))
if controller.get('last_error'):
    label='Earlier startup error (recovered)' if controller_status=='RUNNING after verified recovery' else 'Last error'
    display(HTML('<p><b>'+html.escape(label)+':</b> '+html.escape(controller['last_error'])+'</p>'))
if not reviews:display(HTML('<p>No v5 playing-strength results yet. Verified tactical success does not establish full-game strength.</p>'))
